# Redução de Dimensionalidade

## Tech Challenge Fase 3 - Machine Learning Engineering

**Objetivo:** Aplicar técnicas de redução de dimensionalidade (PCA e t-SNE) para visualização e preparação dos dados para modelagem.

### Estrutura da Análise:
1. Preparação dos Dados
2. PCA (Principal Component Analysis)
3. t-SNE (t-Distributed Stochastic Neighbor Embedding)
4. Comparação e Avaliação
5. Preparação de Features para Aprendizado Supervisionado

In [ ]:
# Imports e Configurações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys

# Adiciona o diretório raiz ao path para importar módulos locais
sys.path.append('..')

# Imports dos módulos locais
from src.unsupervised.dimensionality import (
    apply_pca,
    apply_tsne,
    plot_explained_variance,
    plot_pca_components,
    plot_feature_loadings,
    plot_biplot,
    get_top_features_per_component,
    find_optimal_components,
    prepare_pca_for_supervised
)

from src.unsupervised.evaluate import (
    evaluate_dimensionality_reduction,
    evaluate_pca_reconstruction
)

from src.unsupervised.clustering import prepare_data_for_clustering

# Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Seed para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Cores para gráficos
COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

print('✅ Bibliotecas carregadas com sucesso!')

---
## 1. Preparação dos Dados

In [ ]:
# Carrega os dados processados da EDA
df = pd.read_parquet('../data/processed/flights_processed.parquet')
print(f'Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
df.head()

In [ ]:
# Define as features numéricas para redução de dimensionalidade
# Usamos mais features do que no clustering para explorar correlações

features_dimensionality = [
    'DEPARTURE_DELAY',
    'ARRIVAL_DELAY', 
    'DISTANCE',
    'SCHEDULED_TIME',
    'ELAPSED_TIME',
    'AIR_TIME',
    'TAXI_OUT',
    'TAXI_IN',
    'DAY_OF_WEEK',
    'MONTH'
]

# Verifica se HOUR está disponível (criada na EDA)
if 'HOUR' in df.columns:
    features_dimensionality.append('HOUR')

print(f'Features para análise: {features_dimensionality}')

In [ ]:
# Filtra dados válidos
df_valid = df[df['CANCELLED'] == 0].copy()
df_valid = df_valid.dropna(subset=features_dimensionality)
print(f'Registros válidos: {len(df_valid):,}')

# Amostra para análise (devido ao tamanho do dataset)
SAMPLE_SIZE = 50000
df_sample = df_valid.sample(n=min(SAMPLE_SIZE, len(df_valid)), random_state=RANDOM_STATE)
print(f'Amostra para análise: {len(df_sample):,} registros')

In [ ]:
# Prepara os dados (normalização)
X_scaled, scaler, feature_names = prepare_data_for_clustering(
    df_sample,
    features=features_dimensionality,
    method='standard'
)

print(f'Shape dos dados normalizados: {X_scaled.shape}')
print(f'Features: {feature_names}')

In [ ]:
# Matriz de correlação
df_features = pd.DataFrame(X_scaled, columns=feature_names)
corr_matrix = df_features.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax
)
ax.set_title('Matriz de Correlação das Features', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/unsupervised/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. PCA (Principal Component Analysis)

O PCA é uma técnica linear de redução de dimensionalidade que projeta os dados em um novo espaço de menor dimensão, maximizando a variância explicada.

In [ ]:
# Aplica PCA com todos os componentes para análise de variância
X_pca_full, pca_full, explained_variance_full = apply_pca(
    X_scaled,
    n_components=None  # Todos os componentes
)

print(f'Número total de componentes: {pca_full.n_components_}')
print(f'\nVariância explicada por componente:')
for i, var in enumerate(explained_variance_full):
    print(f'  PC{i+1}: {var*100:.2f}%')

In [ ]:
# Variância acumulada
cumulative_variance = np.cumsum(explained_variance_full)
print('\nVariância acumulada:')
for i, var in enumerate(cumulative_variance):
    print(f'  PC1-PC{i+1}: {var*100:.2f}%')

In [ ]:
# Gráfico de variância explicada
fig = plot_explained_variance(
    pca_full,
    threshold=0.95,
    save_path='../reports/figures/unsupervised/pca_explained_variance.png'
)
plt.show()

In [ ]:
# Encontra número ótimo de componentes para diferentes thresholds
thresholds = [0.80, 0.85, 0.90, 0.95, 0.99]

print('Número de componentes por threshold de variância:')
for threshold in thresholds:
    n_components = find_optimal_components(pca_full, variance_threshold=threshold)
    print(f'  {threshold*100:.0f}% variância: {n_components} componentes')

In [ ]:
# Aplica PCA com número ótimo de componentes (95% variância)
N_COMPONENTS = find_optimal_components(pca_full, variance_threshold=0.95)

X_pca, pca_model, explained_variance = apply_pca(
    X_scaled,
    n_components=N_COMPONENTS
)

print(f'PCA com {N_COMPONENTS} componentes')
print(f'Variância total explicada: {sum(explained_variance)*100:.2f}%')
print(f'\nShape original: {X_scaled.shape}')
print(f'Shape reduzido: {X_pca.shape}')
print(f'Redução: {(1 - X_pca.shape[1]/X_scaled.shape[1])*100:.1f}%')

In [ ]:
# Loadings (contribuição de cada feature para cada componente)
loadings = pd.DataFrame(
    pca_model.components_.T,
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)],
    index=feature_names
)

print('Loadings dos Componentes Principais:')
loadings

In [ ]:
# Visualização dos loadings
fig = plot_feature_loadings(
    pca_model,
    feature_names,
    n_components=min(4, N_COMPONENTS),
    save_path='../reports/figures/unsupervised/pca_loadings.png'
)
plt.show()

In [ ]:
# Top features por componente
top_features = get_top_features_per_component(
    pca_model,
    feature_names,
    n_top=5
)

print('Top 5 features mais importantes por componente:')
for pc, features in top_features.items():
    print(f'\n{pc}:')
    for feat, loading in features:
        direction = '+' if loading > 0 else '-'
        print(f'  {direction} {feat}: {abs(loading):.3f}')

In [ ]:
# Biplot (2 primeiros componentes)
fig = plot_biplot(
    X_pca[:, :2],
    pca_model.components_[:2],
    feature_names,
    explained_variance[:2],
    save_path='../reports/figures/unsupervised/pca_biplot.png'
)
plt.show()

In [ ]:
# Visualização em 2D com PCA colorido por variável de interesse
df_pca = pd.DataFrame(
    X_pca[:, :2],
    columns=['PC1', 'PC2']
)

# Adiciona variáveis categóricas para coloração
df_pca['IS_DELAYED'] = df_sample['IS_DELAYED'].values if 'IS_DELAYED' in df_sample.columns else 0
df_pca['DELAY_CATEGORY'] = df_sample['DELAY_CATEGORY'].values if 'DELAY_CATEGORY' in df_sample.columns else 'Unknown'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por IS_DELAYED
if 'IS_DELAYED' in df_sample.columns:
    scatter = axes[0].scatter(
        df_pca['PC1'],
        df_pca['PC2'],
        c=df_pca['IS_DELAYED'],
        cmap='RdYlGn_r',
        alpha=0.5,
        s=10
    )
    axes[0].set_xlabel(f'PC1 ({explained_variance[0]*100:.1f}%)')
    axes[0].set_ylabel(f'PC2 ({explained_variance[1]*100:.1f}%)')
    axes[0].set_title('PCA - Colorido por IS_DELAYED')
    plt.colorbar(scatter, ax=axes[0], label='Atrasado')

# Por DELAY_CATEGORY
if 'DELAY_CATEGORY' in df_sample.columns:
    categories = df_pca['DELAY_CATEGORY'].unique()
    colors = plt.cm.viridis(np.linspace(0, 1, len(categories)))
    
    for cat, color in zip(categories, colors):
        mask = df_pca['DELAY_CATEGORY'] == cat
        axes[1].scatter(
            df_pca.loc[mask, 'PC1'],
            df_pca.loc[mask, 'PC2'],
            c=[color],
            label=cat,
            alpha=0.5,
            s=10
        )
    axes[1].set_xlabel(f'PC1 ({explained_variance[0]*100:.1f}%)')
    axes[1].set_ylabel(f'PC2 ({explained_variance[1]*100:.1f}%)')
    axes[1].set_title('PCA - Colorido por DELAY_CATEGORY')
    axes[1].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/unsupervised/pca_by_delay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Avaliação da qualidade da reconstrução
reconstruction_metrics = evaluate_pca_reconstruction(
    X_scaled,
    pca_model,
    X_pca
)

print('Métricas de Reconstrução PCA:')
for metric, value in reconstruction_metrics.items():
    print(f'  {metric}: {value:.4f}')

---
## 3. t-SNE (t-Distributed Stochastic Neighbor Embedding)

O t-SNE é uma técnica não-linear de redução de dimensionalidade, especialmente útil para visualização em 2D/3D. Ele preserva estruturas locais dos dados.

In [ ]:
# t-SNE é computacionalmente caro, usamos amostra menor
TSNE_SAMPLE = min(10000, len(X_scaled))

np.random.seed(RANDOM_STATE)
idx_tsne = np.random.choice(len(X_scaled), TSNE_SAMPLE, replace=False)
X_tsne_input = X_scaled[idx_tsne]

print(f'Amostra para t-SNE: {TSNE_SAMPLE} registros')

In [ ]:
# Aplica t-SNE com diferentes valores de perplexity
perplexities = [5, 30, 50]
tsne_results = {}

fig, axes = plt.subplots(1, len(perplexities), figsize=(5*len(perplexities), 5))

for idx, perplexity in enumerate(perplexities):
    print(f'\nExecutando t-SNE com perplexity={perplexity}...')
    
    X_tsne, tsne_model = apply_tsne(
        X_tsne_input,
        n_components=2,
        perplexity=perplexity,
        random_state=RANDOM_STATE
    )
    
    tsne_results[perplexity] = X_tsne
    
    # Colorir por IS_DELAYED se disponível
    if 'IS_DELAYED' in df_sample.columns:
        colors = df_sample.iloc[idx_tsne]['IS_DELAYED'].values
    else:
        colors = 'blue'
    
    scatter = axes[idx].scatter(
        X_tsne[:, 0],
        X_tsne[:, 1],
        c=colors,
        cmap='RdYlGn_r',
        alpha=0.5,
        s=10
    )
    axes[idx].set_title(f't-SNE (perplexity={perplexity})')
    axes[idx].set_xlabel('t-SNE 1')
    axes[idx].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('../reports/figures/unsupervised/tsne_perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# t-SNE final com perplexity ótimo
BEST_PERPLEXITY = 30  # Valor padrão geralmente bom

X_tsne_final, tsne_model_final = apply_tsne(
    X_tsne_input,
    n_components=2,
    perplexity=BEST_PERPLEXITY,
    random_state=RANDOM_STATE
)

print(f'\nt-SNE final com perplexity={BEST_PERPLEXITY}')
print(f'Shape: {X_tsne_final.shape}')

In [ ]:
# Visualização detalhada t-SNE
df_tsne = pd.DataFrame(
    X_tsne_final,
    columns=['tSNE1', 'tSNE2']
)

# Adiciona variáveis para análise
df_tsne['IS_DELAYED'] = df_sample.iloc[idx_tsne]['IS_DELAYED'].values if 'IS_DELAYED' in df_sample.columns else 0
df_tsne['DELAY_CATEGORY'] = df_sample.iloc[idx_tsne]['DELAY_CATEGORY'].values if 'DELAY_CATEGORY' in df_sample.columns else 'Unknown'
df_tsne['ARRIVAL_DELAY'] = df_sample.iloc[idx_tsne]['ARRIVAL_DELAY'].values

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Por IS_DELAYED
scatter1 = axes[0].scatter(
    df_tsne['tSNE1'],
    df_tsne['tSNE2'],
    c=df_tsne['IS_DELAYED'],
    cmap='RdYlGn_r',
    alpha=0.5,
    s=10
)
axes[0].set_title('t-SNE - Colorido por IS_DELAYED')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
plt.colorbar(scatter1, ax=axes[0], label='Atrasado')

# Por DELAY_CATEGORY
categories = df_tsne['DELAY_CATEGORY'].unique()
colors = plt.cm.viridis(np.linspace(0, 1, len(categories)))

for cat, color in zip(sorted(categories), colors):
    mask = df_tsne['DELAY_CATEGORY'] == cat
    axes[1].scatter(
        df_tsne.loc[mask, 'tSNE1'],
        df_tsne.loc[mask, 'tSNE2'],
        c=[color],
        label=cat,
        alpha=0.5,
        s=10
    )
axes[1].set_title('t-SNE - Colorido por DELAY_CATEGORY')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(loc='best', fontsize=8)

# Por magnitude do atraso
scatter3 = axes[2].scatter(
    df_tsne['tSNE1'],
    df_tsne['tSNE2'],
    c=np.clip(df_tsne['ARRIVAL_DELAY'], -60, 180),  # Clip para visualização
    cmap='RdYlBu_r',
    alpha=0.5,
    s=10
)
axes[2].set_title('t-SNE - Colorido por ARRIVAL_DELAY')
axes[2].set_xlabel('t-SNE 1')
axes[2].set_ylabel('t-SNE 2')
plt.colorbar(scatter3, ax=axes[2], label='Atraso (min)')

plt.tight_layout()
plt.savefig('../reports/figures/unsupervised/tsne_detailed.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Comparação PCA vs t-SNE

In [ ]:
# Avaliação da qualidade da redução de dimensionalidade
# Usa a mesma amostra para comparação
X_pca_sample = X_pca[idx_tsne]

pca_eval = evaluate_dimensionality_reduction(
    X_tsne_input,
    X_pca_sample[:, :2],  # Apenas 2 componentes para comparar com t-SNE
    method='pca'
)

tsne_eval = evaluate_dimensionality_reduction(
    X_tsne_input,
    X_tsne_final,
    method='tsne'
)

print('Avaliação PCA (2 componentes):')
for metric, value in pca_eval.items():
    print(f'  {metric}: {value:.4f}')

print('\nAvaliação t-SNE:')
for metric, value in tsne_eval.items():
    print(f'  {metric}: {value:.4f}')

In [ ]:
# Visualização comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA
if 'IS_DELAYED' in df_sample.columns:
    colors_pca = df_sample.iloc[idx_tsne]['IS_DELAYED'].values
else:
    colors_pca = 'blue'

scatter1 = axes[0].scatter(
    X_pca_sample[:, 0],
    X_pca_sample[:, 1],
    c=colors_pca,
    cmap='RdYlGn_r',
    alpha=0.5,
    s=10
)
axes[0].set_title(f'PCA (Variância: {sum(explained_variance[:2])*100:.1f}%)')
axes[0].set_xlabel(f'PC1 ({explained_variance[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({explained_variance[1]*100:.1f}%)')
plt.colorbar(scatter1, ax=axes[0], label='Atrasado')

# t-SNE
scatter2 = axes[1].scatter(
    X_tsne_final[:, 0],
    X_tsne_final[:, 1],
    c=colors_pca,
    cmap='RdYlGn_r',
    alpha=0.5,
    s=10
)
axes[1].set_title(f't-SNE (perplexity={BEST_PERPLEXITY})')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=axes[1], label='Atrasado')

plt.suptitle('Comparação: PCA vs t-SNE', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/unsupervised/pca_vs_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tabela comparativa
comparison_table = pd.DataFrame({
    'Métrica': ['Spearman Correlation', 'Trustworthiness', 'Dimensões de Saída', 'Interpretável', 'Escalável'],
    'PCA': [f"{pca_eval['spearman_correlation']:.3f}", f"{pca_eval['trustworthiness']:.3f}", '2', 'Sim', 'Sim'],
    't-SNE': [f"{tsne_eval['spearman_correlation']:.3f}", f"{tsne_eval['trustworthiness']:.3f}", '2', 'Não', 'Não']
})

print('Tabela Comparativa PCA vs t-SNE:')
comparison_table

---
## 5. Preparação de Features para Aprendizado Supervisionado

Aqui preparamos os componentes PCA para uso em modelos supervisionados, seguindo boas práticas para evitar data leakage.

In [ ]:
# Carrega dados completos para preparação
df_full = pd.read_parquet('../data/processed/flights_processed.parquet')
df_full = df_full[df_full['CANCELLED'] == 0].copy()
df_full = df_full.dropna(subset=features_dimensionality)

print(f'Dados completos para modelagem: {len(df_full):,} registros')

In [ ]:
# Prepara PCA para aprendizado supervisionado (evita data leakage)
# Esta função faz o split, normaliza, e aplica PCA corretamente

X = df_full[features_dimensionality].values

# Target para classificação
if 'IS_DELAYED' in df_full.columns:
    y = df_full['IS_DELAYED'].values
else:
    y = (df_full['ARRIVAL_DELAY'] >= 15).astype(int).values

# Prepara os dados com PCA (fit apenas no treino)
pca_result = prepare_pca_for_supervised(
    X=X,
    y=y,
    n_components=0.95,  # 95% da variância
    test_size=0.2,
    val_size=0.1,
    random_state=RANDOM_STATE
)

print('Conjuntos preparados:')
print(f"  X_train_pca: {pca_result['X_train_pca'].shape}")
print(f"  X_val_pca: {pca_result['X_val_pca'].shape}")
print(f"  X_test_pca: {pca_result['X_test_pca'].shape}")
print(f"  y_train: {pca_result['y_train'].shape}")
print(f"  y_val: {pca_result['y_val'].shape}")
print(f"  y_test: {pca_result['y_test'].shape}")
print(f"\n  Variância explicada: {sum(pca_result['pca'].explained_variance_ratio_)*100:.2f}%")
print(f"  Número de componentes: {pca_result['pca'].n_components_}")

In [ ]:
# Salva os transformers para uso posterior
import joblib

joblib.dump(pca_result['scaler'], '../models/transformers/scaler_pca.joblib')
joblib.dump(pca_result['pca'], '../models/transformers/pca_model.joblib')

print('✅ Transformers salvos:')
print('  - models/transformers/scaler_pca.joblib')
print('  - models/transformers/pca_model.joblib')

In [ ]:
# Salva os dados transformados para uso em notebooks de modelagem
np.savez(
    '../data/processed/pca_data.npz',
    X_train_pca=pca_result['X_train_pca'],
    X_val_pca=pca_result['X_val_pca'],
    X_test_pca=pca_result['X_test_pca'],
    y_train=pca_result['y_train'],
    y_val=pca_result['y_val'],
    y_test=pca_result['y_test']
)

print('✅ Dados PCA salvos em data/processed/pca_data.npz')

---
## Conclusões

### Principais Descobertas:

1. **PCA:**
   - Com N componentes, conseguimos explicar ~95% da variância
   - Os primeiros componentes capturam principalmente variações em tempo/distância de voo e atrasos
   - O PCA é útil para reduzir dimensionalidade mantendo interpretabilidade

2. **t-SNE:**
   - Melhor para visualização de estruturas não-lineares
   - Revela clusters de voos com comportamentos similares
   - Não é recomendado para pré-processamento de features (não preserva distâncias globais)

3. **Comparação:**
   - PCA: Rápido, interpretável, escalável - ideal para pré-processamento
   - t-SNE: Melhor visualização de clusters - ideal para exploração

### Uso Recomendado:

- **Para Modelagem Supervisionada:** Usar componentes PCA como features
- **Para Visualização:** Usar t-SNE para entender estrutura dos dados
- **Para Clustering:** PCA para reduzir dimensionalidade antes de aplicar clustering